In [ ]:
import sys
import os
from typing import NamedTuple
import math
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import pickle

from sklearn.model_selection import TimeSeriesSplit, StratifiedKFold
import jax


sys.path.append(os.path.join(os.getcwd(), '../'))
from utils_worm import artificializer
from utils_worm import read_utils

sys.path.append(os.path.join(os.getcwd(), '../../../../'))
from utils import plot_utils
from utils import train_utils
from models import bom_ss as bom
from models import hgm_ss as hgm
from models import hgm as hgm_gen


# jax.config.update("jax_enable_x64", True)

In [ ]:
import importlib
importlib.reload(artificializer)
importlib.reload(read_utils)
importlib.reload(plot_utils)
importlib.reload(train_utils)
importlib.reload(bom)
importlib.reload(hgm)

In [ ]:
def seperate_predictions(models_spn_man_rec_res_pres_resss_bss, columns):
	models_spn = [
		pd.DataFrame(data=np.array(split[0])[None], columns=columns)
		for split in models_spn_man_rec_res_pres_resss_bss
	]
	models_man = [
		pd.DataFrame(data=np.array(split[1]), columns=[f'M{i}' for i in range(np.array(split[1]).shape[1])])
		for split in models_spn_man_rec_res_pres_resss_bss
	]
	(models_rec,
	 models_res,
	 models_pres) = tuple([
		pd.DataFrame(data=np.array(split[m]), columns=columns)
		for split in models_spn_man_rec_res_pres_resss_bss
	] for m in [2, 3, 4])
	(models_resss,
	 models_bss) = tuple(
		[
			pd.DataFrame(data=np.array(split[m]), columns=[f'SS{j}' for j in range(np.array(split[m]).shape[1])]) for split in models_spn_man_rec_res_pres_resss_bss
		] for m in [5, 6]
	)

	return models_spn, models_man, models_rec, models_res, models_pres, models_resss, models_bss

In [ ]:
manual_hyperparameters = {
	'dimensions': 2,
	'loss_man_lambda': 0.0,
	'loss_res_lambda': 1.0 - 0.0,
	'vb_reg_lambda': 0.0,
	'wm_reg_lambda': 0.0,
}

ll = f'll{int(manual_hyperparameters['loss_man_lambda'] * 100):02d}'

with open('../sweep/hyperparams.pkl', 'rb') as f:
    swept_hyperparameters = pickle.load(f)

bom_reg_lambdas = {
	# 'vb_reg_lambda': manual_hyperparameters['vb_reg_lambda'],
	'vb_reg_lambda': swept_hyperparameters[ll]['bom_ss']['means']['vb_reg_lambda']['geometric'],
}
hgm_reg_lambdas = {
	# 'vb_reg_lambda': manual_hyperparameters['vb_reg_lambda'],
	'vb_reg_lambda': swept_hyperparameters[ll]['hgm_ss']['means']['vb_reg_lambda']['geometric'],
	# 'wm_reg_lambda': manual_hyperparameters['wm_reg_lambda'],
	'wm_reg_lambda': swept_hyperparameters[ll]['hgm_ss']['means']['wm_reg_lambda']['geometric'],
}

organism = 'worm-2025-fieseler'

In [ ]:
artificial = True
gcamp = True
gfp = True

In [ ]:
def run_and_save(traces, behavior, states, model, model_params, save_path, with_full=True):

	# predict validation set scores & save them
	splits = list(StratifiedKFold(n_splits=5, shuffle=False).split(np.zeros(traces.shape[0]), states[['REV']].values))

	model_spn_man_rec_res_pres_resss_bss = [
		model.score_fn(params, traces.values[eval], behavior.values[eval]) for params, (train, eval) in zip(model_params, splits)
	]
	(model_spn,
	 model_man,
	 model_rec,
	 model_res,
	 model_pres,
	 model_resss,
	 model_bss) = seperate_predictions(model_spn_man_rec_res_pres_resss_bss, traces.columns)
	train_utils.save_predictions(
		(model_spn,
		 None,
		 None,
		 None,
		 None,
		 None,
		 None),
		save_path=f'{save_path}/validation')

	if with_full:
		# predict full traces & save them
		model_spn_man_rec_res_pres_resss_bss = [
			model.score_fn(params, traces.values, behavior.values) for params in model_params
		]
		(model_spn,
		 model_man,
		 model_rec,
		 model_res,
		 model_pres,
		 model_resss,
		 model_bss) = seperate_predictions(model_spn_man_rec_res_pres_resss_bss, traces.columns)
		train_utils.save_predictions(
			(None,
			 model_man,
			 model_rec,
			 model_res,
			 model_pres,
			 model_resss,
			 model_bss),
			save_path=f'{save_path}/full')

In [ ]:
if gcamp:
	file_ids = list(range(0, 33+1))
	# file_ids = [0,   # minrevs
	# 			  1,   # minswitch
	# 			  7,   # maxswitch
	# 			  11,  # median
	# 			  12,  # maxrevs
	# 			  17,  # mean
	# 			  19,  # paper
	# 			  21]  # clean

	path = os.path.join(read_utils.get_data_path(), 'exported_data_gcamp')

	bom_load_name = os.path.join(organism, ll, f'ss_bom_gcamp_vb-{bom_reg_lambdas['vb_reg_lambda']:.2E}')
	hgm_load_name = os.path.join(organism, ll, f'ss_hgm_gcamp_vb-{hgm_reg_lambdas['vb_reg_lambda']:.2E}_wm-{hgm_reg_lambdas['wm_reg_lambda']:.2E}')

	for file_id in file_ids:
		# read
		(filename,
		 traces,
		 continuous_behaviors,
		 discrete_states, eigenworms,
		 kymograph) = read_utils.get_data(path, file_id)

		# clear nan values
		mask = eigenworms.notna().all(axis=1)
		traces = traces[mask]
		#continuous_behaviors = continuous_behaviors[mask]
		eigenworms = eigenworms[mask]
		discrete_states = discrete_states[mask]
		#kymograph = kymograph[mask]

		# naive ID unknown neurons
		traces.columns = pd.Index([
			f'{file_id:02d}_{n_id:03d}' if empty else label for n_id, (label, empty) in enumerate(zip(traces.columns, traces.columns == ''))
		])

		# read model, run & save
		model_params = train_utils.load_params(bom_load_name, [filename])[0]
		run_and_save(traces, eigenworms, discrete_states, bom, model_params,
					 save_path=f'../../out/predictions/gcamp/{filename}/bom_ss', with_full=True)

		model_params = train_utils.load_params(hgm_load_name, [filename])[0]
		run_and_save(traces, eigenworms, discrete_states, hgm, model_params,
					 save_path=f'../../out/predictions/gcamp/{filename}/hgm_ss', with_full=True)

In [ ]:
if gfp:
	file_ids = list(range(0, 7+1))

	path = os.path.join(read_utils.get_data_path(), 'exported_data_gfp')

	bom_load_name = os.path.join(organism, ll, f'ss_bom_gfp_vb-{bom_reg_lambdas['vb_reg_lambda']:.2E}')
	hgm_load_name = os.path.join(organism, ll, f'ss_hgm_gfp_vb-{hgm_reg_lambdas['vb_reg_lambda']:.2E}_wm-{hgm_reg_lambdas['wm_reg_lambda']:.2E}')

	for file_id in file_ids:
		# read
		(filename,
		 traces,
		 continuous_behaviors,
		 discrete_states, eigenworms,
		 kymograph) = read_utils.get_data(path, file_id)

		# clear nan values
		mask = eigenworms.notna().all(axis=1)
		traces = traces[mask]
		#continuous_behaviors = continuous_behaviors[mask]
		eigenworms = eigenworms[mask]
		discrete_states = discrete_states[mask]
		#kymograph = kymograph[mask]

		# naive ID unknown neurons
		traces.columns = pd.Index([
			f'{file_id:02d}_{n_id:03d}' if empty else label for n_id, (label, empty) in enumerate(zip(traces.columns, traces.columns == ''))
		])

		# read model, run & save
		model_params = train_utils.load_params(bom_load_name, [filename])[0]
		run_and_save(traces, eigenworms, discrete_states, bom, model_params,
					 save_path=f'../../out/predictions/gfp/{filename}/bom_ss', with_full=True)

		model_params = train_utils.load_params(hgm_load_name, [filename])[0]
		run_and_save(traces, eigenworms, discrete_states, hgm, model_params,
					 save_path=f'../../out/predictions/gfp/{filename}/hgm_ss', with_full=True)

In [ ]:
if artificial:
	file_ids = list(range(0, 9+1))
	dims_generation = 2
	weights = (0.1, -0.25)
	selection_bhl = ['REV', 'VENTRAL_TURN', 'DORSAL_TURN', 'PAUSE']

	path = os.path.join(read_utils.get_data_path(), 'exported_data_gcamp')

	bom_load_name = os.path.join(organism, ll, f'ss_bom_art_vb-{bom_reg_lambdas['vb_reg_lambda']:.2E}')
	hgm_load_name = os.path.join(organism, ll, f'ss_hgm_art_vb-{hgm_reg_lambdas['vb_reg_lambda']:.2E}_wm-{hgm_reg_lambdas['wm_reg_lambda']:.2E}')

	for file_id in file_ids:
		# read gcamp data
		(filename,
		 traces,
		 continuous_behaviors,
		 discrete_states, eigenworms,
		 kymograph) = read_utils.get_data(path, file_id)

		# clear nan values
		mask = eigenworms.notna().all(axis=1)
		traces = traces[mask]
		#continuous_behaviors = continuous_behaviors[mask]
		eigenworms = eigenworms[mask]
		discrete_states = discrete_states[mask]
		#kymograph = kymograph[mask]

		# naive ID unknown neurons
		traces.columns = pd.Index([
			f'{file_id:02d}_{n_id:03d}' if empty else label for n_id, (label, empty) in enumerate(zip(traces.columns, traces.columns == ''))
		])

		# generate artificial data
		(filename,
		 neural_modes,
		 traces_reconstructed,
		 traces_residual,
		 traces,
		 params,
		 params_cca) = artificializer.artificialize(weights, hgm_gen,
													filename,
													traces.columns,
													eigenworms,
													discrete_states[selection_bhl],
													n_dimensions=dims_generation)

		# read model, run & save
		model_params = train_utils.load_params(bom_load_name, [filename])[0]
		run_and_save(traces, eigenworms, discrete_states, bom, model_params,
					 save_path=f'../../out/predictions/art/{filename}/bom_ss', with_full=True)

		model_params = train_utils.load_params(hgm_load_name, [filename])[0]
		run_and_save(traces, eigenworms, discrete_states, hgm, model_params,
					 save_path=f'../../out/predictions/art/{filename}/hgm_ss', with_full=True)